# 第三章: 立体视觉点云重建

## 学习目标
- 掌握从图像序列重建三维点云的计算原理和编程实现
- 理解运动恢复结构（Structure from Motion, SFM）的基本流程
- 掌握特征点匹配、相机位姿估计、三角化重建的实现

## 编程实践
下载 SFM 数据集的前两张图像进行点云重建，包括特征点提取、匹配、估计相机相对位姿、三角化匹配点的三维坐标。保存点云坐标到 TXT 文件，用开源软件显示。

> **要求**：除图像读写、矩阵计算、特征点提取&匹配部分可调用 OpenCV/Numpy 函数外，其余代码都是自己手写，不能调用其他函数库。

---
## 1. 运动恢复结构 (SFM)

### 1.1 什么是 SFM？
SFM（Structure from Motion）是从多张不同视角的二维图像恢复场景三维结构的技术。核心思想：
- **运动**：相机在空间中的位姿发生变化
- **结构**：场景物体的三维结构

### 1.2 SFM 基本流程
```
图像序列
  │
  ├──→ 特征点提取 (SIFT/ORB)
  │
  ├──→ 特征点匹配 (跨视图匹配)
  │
  ├──→ 相机位姿估计 (本质矩阵 E / 基础矩阵 F)
  │       │
  │       └──→ 分解 E 得到 R 和 t
  │
  ├──→ 三角化 (Triangulation)
  │       │
  │       └──→ 从 2D 对应点恢复 3D 坐标
  │
  └──→ 光束法平差 (Bundle Adjustment)
          │
          └──→ 联合优化所有 3D 点和相机参数
```

### 1.3 本章范围
本章实现 SFM 的核心步骤：两视图重建
- 特征提取与匹配
- 本质矩阵估计与相机位姿恢复
- 三角化获取 3D 点云
不包含光束法平差（留给进阶课程）

---
## 2. 对极几何基础

### 2.1 基础矩阵 F 和本质矩阵 E

**基础矩阵 F**（未标定）：
```
x'^T F x = 0
```
- 描述两幅图像对应点的几何关系
- 仅依赖相机内参数和位姿
- F 秩为 2，有 7 个自由度

**本质矩阵 E**（已标定）：
```
E = K'^T F K
x'^T E x = 0
```
- 内参数已知时使用
- 包含旋转 R 和平移 t 的信息
- E 秩为 2，有 5 个自由度

### 2.2 本质矩阵的分解
给定 E，可以分解得到 R 和 t：
```
E = [t]_× R
```

其中 [t]_× 是平移向量的反对称矩阵：
```
[t]_× = | 0  -tz  ty |
        | tz   0  -tx |
        | -ty  tx   0 |
```

### 2.3 由 E 恢复 R 和 t
1. 对 E 做 SVD 分解: E = U Σ V^T
2. 构造 W 和 Z 矩阵
3. 计算 R = U W V^T (或 U W^T V^T)
4. 计算 t = ±U 的第三列

会得到 4 种组合，只有一种满足两个相机位姿下点都在前方。

---
## 3. 三角化重建

### 3.1 三角化原理
已知：
- 两张相机的投影矩阵 P1, P2
- 一个 3D 点在两图像中的 2D 投影 x1, x2

求解 3D 点 X，使得：
```
x1 ~ P1 X
x2 ~ P2 X
```

### 3.2 线性三角化
对于每个对应点对 (x1, x2)，构造线性方程组：
```
A = | P1[0] - x1[0]*P1[2] |   | X |   | 0 |
    | P1[1] - x1[1]*P1[2] | × | Y | = | 0 |
    | P2[0] - x2[0]*P2[2] |   | Z |   | 0 |
    | P2[1] - x2[1]*P2[2] |   | W |   | 0 |
```

用 SVD 求解 AX = 0（齐次坐标），得到 3D 点。

### 3.3 投影矩阵
```
P1 = K × [I|0]  (第一个相机)
P2 = K × [R|t]  (第二个相机)
```

其中 K 是相机内参数矩阵，R 和 t 是相机位姿。

In [ ]:
# ===== 环境设置 =====
import os
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# ===== 中文路径兼容的图像读写函数 =====
# OpenCV 在 Windows 中文路径下 imread/imwrite 会失败
def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    import numpy as np
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False


from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {os.getcwd()}")



In [ ]:
# ===== 第一步: 读取立体图像对 =====
img1 = cv_imread('stereo_image_1.jpg')
img2 = cv_imread('stereo_image_2.jpg')

if img1 is None or img2 is None:
    raise FileNotFoundError("无法读取立体图像对, 请确保 stereo_image_1.jpg 和 stereo_image_2.jpg 存在")

h, w = img1.shape[:2]
print(f"图像尺寸: {w}x{h}")
print(f"图像 1: stereo_image_1.jpg")
print(f"图像 2: stereo_image_2.jpg")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0].set_title('相机 1 (View 1)', fontsize=12)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[1].set_title('相机 2 (View 2)', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()

# 转换为灰度图
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)


In [ ]:
# ===== 第二步: 特征提取与匹配 =====

# 使用 SIFT 提取特征
sift = cv2.SIFT_create(nfeatures=2000)

kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

print(f"图像1关键点: {len(kp1)}")
print(f"图像2关键点: {len(kp2)}")

# FLANN 匹配
flann = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), {})
matches = flann.knnMatch(des1, des2, k=2)

# Lowe's Ratio Test
good_matches = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good_matches.append(m)

print(f"良好匹配点: {len(good_matches)}")

# 获取匹配点坐标 (像素坐标)
pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches])
pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches])

# 可视化匹配
match_img = cv2.drawMatches(img1, kp1, img2, kp2, good_matches[:50], None,
                           flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(14, 5))
plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
plt.title(f'特征匹配 ({len(good_matches)} 对)', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ===== 第三步: 估计本质矩阵 E =====

# 假设相机内参数 (使用图像中心和焦距的合理估计)
# 实际中应通过标定得到
fx = fy = w * 1.2  # 假设焦距
cx, cy = w / 2, h / 2

K = np.float64([
    [fx, 0, cx],
    [0, fy, cy],
    [0, 0, 1]
])

print(f"假设的内参数矩阵 K:")
print(f"  fx = fy = {fx:.1f}")
print(f"  cx = {cx:.1f}, cy = {cy:.1f}")

# 用 RANSAC 估计基础矩阵 F
F, F_inliers = cv2.findFundamentalMat(pts1, pts2, cv2.FM_RANSAC, 1.0, 0.99)

if F is not None:
    print(f"\n基础矩阵 F (3x3):")
    print(F)
    
    # 计算内点数
    inlier_mask = F_inliers.ravel().astype(bool)
    num_inliers = np.sum(inlier_mask)
    print(f"\nRANSAC 内点: {num_inliers} / {len(pts1)}")
    print(f"内点率: {num_inliers/len(pts1)*100:.1f}%")
    
    # 用内点重新估计
    pts1_inlier = pts1[inlier_mask]
    pts2_inlier = pts2[inlier_mask]
    
    # 由 F 计算 E: E = K^T F K
    E = K.T @ F @ K
    print(f"\n本质矩阵 E (3x3):")
    print(E)
    
    # 验证 E 的秩应为 2
    U, S, Vt = np.linalg.svd(E)
    print(f"\nE 的奇异值: {S}")
    print(f"(前两个应非零, 第三个应接近 0)")
else:
    print("无法估计基础矩阵, 匹配点可能不足")
    pts1_inlier = pts1
    pts2_inlier = pts2
    E = K.T @ F @ K if F is not None else None

In [ ]:
# ===== 第四步: 手写分解本质矩阵 E → 相机位姿 =====

def decompose_essential_matrix(E):
    """
    手写分解本质矩阵 E 得到旋转 R 和平移 t
    
    根据 Hartley & Zisserman 的方法:
    E = [t]_× R
    
    参数:
        E: 3x3 本质矩阵
    
    返回:
        R: 3x3 旋转矩阵
        t: 3x1 平移向量 (单位化)
    """
    # ===== SVD 分解 =====
    U, S, Vt = np.linalg.svd(E)
    
    # 确保 SVD 的符号一致性
    if np.linalg.det(U) < 0:
        U[:, 2] *= -1
    if np.linalg.det(Vt) < 0:
        Vt[2, :] *= -1
    
    # ===== 构造 W 和 Z =====
    # W = | 0  -1  0 |    Z = | 0   1  0 |
    #     | 1   0  0 |        | -1  0  0 |
    #     | 0   0  1 |        | 0   0  0 |
    W = np.float64([
        [0, -1, 0],
        [1, 0, 0],
        [0, 0, 1]
    ])
    
    # ===== 计算四个可能的解 =====
    # 公式: R = U W V^T 或 U W^T V^T
    #       t = U 的第三列 (±)
    
    R1 = U @ W @ Vt
    R2 = U @ W.T @ Vt
    
    t1 = U[:, 2].reshape(3, 1)    # +t
    t2 = -U[:, 2].reshape(3, 1)   # -t
    
    # 返回所有 4 种组合
    solutions = [
        (R1, t1),
        (R1, t2),
        (R2, t1),
        (R2, t2)
    ]
    
    return solutions

def select_correct_pose(solutions, pts1, pts2, K):
    """
    从 4 组解中选择正确的相机位姿
    准则: 大多数 3D 点应在两个相机前方
    """
    best_solution = None
    best_count = -1
    
    for R, t in solutions:
        count = 0
        
        # 投影矩阵
        P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
        P2 = K @ np.hstack([R, t])
        
        # 对每个匹配点进行三角化并检查深度
        for i in range(len(pts1)):
            # 三角化 (线性方法)
            X = linear_triangulation(pts1[i], pts2[i], P1, P2)
            
            if X is not None:
                # 检查在两个相机坐标系下的深度
                X_cam1 = np.hstack([np.eye(3), np.zeros((3, 1))]) @ X
                X_cam2 = np.hstack([R, t]) @ X
                
                if X_cam1[2] > 0 and X_cam2[2] > 0:
                    count += 1
        
        if count > best_count:
            best_count = count
            best_solution = (R, t)
    
    return best_solution

def linear_triangulation(pt1, pt2, P1, P2):
    """
    线性三角化 (DLT 方法)
    
    给定两个相机的投影矩阵和匹配点, 求 3D 点
    """
    x1, y1 = pt1[0], pt1[1]
    x2, y2 = pt2[0], pt2[1]
    
    # 构造 4x4 线性方程组
    # 从 x1*P1[2] - P1[0] = 0 和 y1*P1[2] - P1[1] = 0 等
    A = np.zeros((4, 4), dtype=np.float64)
    
    # 第一张相机的约束
    A[0] = x1 * P1[2] - P1[0]  # x1 * P1_row3 - P1_row1
    A[1] = y1 * P1[2] - P1[1]  # y1 * P1_row3 - P1_row2
    
    # 第二张相机的约束
    A[2] = x2 * P2[2] - P2[0]
    A[3] = y2 * P2[2] - P2[1]
    
    # SVD 求解 AX = 0
    U, S, Vt = np.linalg.svd(A)
    X = Vt[-1]  # 最小奇异值对应的向量
    
    # 齐次坐标 → 非齐次
    if abs(X[3]) < 1e-10:
        return None
    X = X / X[3]
    
    return X

# 执行分解
print("=" * 50)
print("分解本质矩阵 E → 相机位姿")
print("=" * 50)

solutions = decompose_essential_matrix(E)
print(f"得到 4 组可能的 (R, t) 解")

# 选择正确的解
R, t = select_correct_pose(solutions, pts1_inlier, pts2_inlier, K)

print(f"\n选择的旋转矩阵 R:")
print(R)
print(f"\n选择的平移向量 t:")
print(t)

# 验证 R 是有效旋转矩阵
print(f"\n验证 R:")
print(f"  det(R) = {np.linalg.det(R):.6f} (应为 1)")
print(f"  R·R^T 对角线 = {np.diag(R @ R.T)} (应为 1)")

# 可视化两个相机位姿
print(f"\n相机 1: 原点 (0, 0, 0), 朝向 (0, 0, 1)")
print(f"相机 2: 位置 ({t[0,0]:.3f}, {t[1,0]:.3f}, {t[2,0]:.3f})")
print(f"       朝向: R · [0,0,1]^T = {R[:, 2]}")

In [ ]:
# ===== 第五步: 三角化重建 3D 点云 =====

# 构造投影矩阵
P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
P2 = K @ np.hstack([R, t])

print(f"投影矩阵 P1 (3x4):")
print(P1)
print(f"\n投影矩阵 P2 (3x4):")
print(P2)

# 对所有内点进行三角化
points_3d = []
valid_indices = []

for i in range(len(pts1_inlier)):
    X = linear_triangulation(pts1_inlier[i], pts2_inlier[i], P1, P2)
    
    if X is not None:
        # 检查深度 (两个相机前方)
        X_cam1 = np.hstack([np.eye(3), np.zeros((3, 1))]) @ X
        X_cam2 = np.hstack([R, t]) @ X
        
        if X_cam1[2] > 0 and X_cam2[2] > 0:
            points_3d.append(X[:3])
            valid_indices.append(i)

points_3d = np.array(points_3d)
print(f"\n三角化得到 {len(points_3d)} 个有效 3D 点")

# 计算重投影误差
print(f"\n重投影误差:")
errors = []
for i, idx in enumerate(valid_indices):
    X_homo = np.hstack([points_3d[i], 1.0])  # 齐次坐标
    
    # 投影回图像1
    x1_proj = P1 @ X_homo
    x1_proj = x1_proj / x1_proj[2]  # 齐次归一
    
    # 投影回图像2
    x2_proj = P2 @ X_homo
    x2_proj = x2_proj / x2_proj[2]
    
    # 计算误差
    err1 = np.sqrt((x1_proj[0] - pts1_inlier[idx][0])**2 + 
                   (x1_proj[1] - pts1_inlier[idx][1])**2)
    err2 = np.sqrt((x2_proj[0] - pts2_inlier[idx][0])**2 + 
                   (x2_proj[1] - pts2_inlier[idx][1])**2)
    errors.append((err1 + err2) / 2)

print(f"  平均重投影误差: {np.mean(errors):.4f} 像素")
print(f"  最大重投影误差: {np.max(errors):.4f} 像素")
print(f"  最小重投影误差: {np.min(errors):.4f} 像素")

# 重投影误差分布
plt.figure(figsize=(8, 4))
plt.hist(errors, bins=30, color='steelblue', edgecolor='black')
plt.xlabel('重投影误差 (像素)')
plt.ylabel('频数')
plt.title('三角化重投影误差分布')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 第六步: 3D 点云可视化 =====

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# 绘制 3D 点云
xs = points_3d[:, 0]
ys = points_3d[:, 1]
zs = points_3d[:, 2]

# 归一化用于可视化
ax.scatter(xs, ys, zs, c='blue', s=20, alpha=0.6, depthshade=True)

# 绘制相机位置
ax.scatter([0], [0], [0], c='red', s=100, marker='s', label='相机 1', zorder=5)
ax.scatter([t[0,0]], [t[1,0]], [t[2,0]], c='green', s=100, marker='s', label='相机 2', zorder=5)

# 绘制相机朝向
scale = 0.5
ax.quiver(0, 0, 0, scale * 0, scale * 0, scale * 1, color='red', linewidth=2)
ax.quiver(t[0,0], t[1,0], t[2,0],
          scale * R[0, 2], scale * R[1, 2], scale * R[2, 2],
          color='green', linewidth=2)

ax.set_xlabel('X', fontsize=11)
ax.set_ylabel('Y', fontsize=11)
ax.set_zlabel('Z', fontsize=11)
ax.set_title(f'3D 点云重建 ({len(points_3d)} 个点)', fontsize=13)
ax.legend(fontsize=10)

# 调整视角
ax.view_init(elev=25, azim=45)
plt.tight_layout()
plt.show()

# 从不同角度查看
fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': '3d'})
views = [(25, 45), (0, 90), (90, 0)]  # (elev, azim)
titles = ['等轴视图', '侧视图', '俯视图']

for ax, (elev, azim), title in zip(axes, views, titles):
    ax.scatter(xs, ys, zs, c='steelblue', s=15, alpha=0.7)
    ax.scatter([0], [0], [0], c='red', s=80, marker='s')
    ax.scatter([t[0,0]], [t[1,0]], [t[2,0]], c='green', s=80, marker='s')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.view_init(elev=elev, azim=azim)

plt.suptitle('3D 点云 - 多视角观察', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ===== 第七步: 保存点云到 TXT 文件 =====

# 保存点云坐标
output_file = 'point_cloud.txt'
with open(output_file, 'w') as f:
    # 写入点云格式说明
    f.write("# 3D 点云坐标 (X, Y, Z)\n")
    f.write(f"# 点数: {len(points_3d)}\n")
    f.write("# 格式: X Y Z\n")
    f.write("#" + "=" * 50 + "\n")
    
    # 写入每个点的坐标
    for point in points_3d:
        f.write(f"{point[0]:.6f} {point[1]:.6f} {point[2]:.6f}\n")

print(f"点云已保存: {output_file}")
print(f"点数: {len(points_3d)}")

# 同时保存相机参数
params_file = 'camera_params.txt'
with open(params_file, 'w') as f:
    f.write("# 相机内参数矩阵 K\n")
    for row in K:
        f.write(f"{row[0]:.6f} {row[1]:.6f} {row[2]:.6f}\n")
    
    f.write("\n# 相机 2 旋转矩阵 R\n")
    for row in R:
        f.write(f"{row[0]:.6f} {row[1]:.6f} {row[2]:.6f}\n")
    
    f.write("\n# 相机 2 平移向量 t\n")
    f.write(f"{t[0,0]:.6f}\n{t[1,0]:.6f}\n{t[2,0]:.6f}\n")

print(f"相机参数已保存: {params_file}")

# 保存匹配点对应关系
matches_file = 'matches_3d.txt'
with open(matches_file, 'w') as f:
    f.write("# 匹配点对应关系 (像素坐标 → 3D 坐标)\n")
    f.write("# 格式: x1 y1 x2 y2 X Y Z\n")
    for i, idx in enumerate(valid_indices):
        p1 = pts1_inlier[idx]
        p2 = pts2_inlier[idx]
        p3d = points_3d[i]
        f.write(f"{p1[0]:.2f} {p1[1]:.2f} {p2[0]:.2f} {p2[1]:.2f} {p3d[0]:.6f} {p3d[1]:.6f} {p3d[2]:.6f}\n")

print(f"匹配关系已保存: {matches_file}")

print(f"\n" + "=" * 50)
print("输出文件:")
print(f"  1. {output_file} - 3D 点云坐标")
print(f"  2. {params_file} - 相机参数")
print(f"  3. {matches_file} - 2D-3D 对应关系")
print(f"\n可以使用 CloudCompare 或 MeshLab 打开 point_cloud.txt 查看点云")
print("=" * 50)

---
## 4. 本章总结

### 核心知识点
1. **SFM 流程**：特征提取 → 匹配 → 位姿估计 → 三角化
2. **对极几何**：基础矩阵 F、本质矩阵 E
   - F: x'^T F x = 0（7 自由度，秩 2）
   - E: [t]_× R（5 自由度，秩 2）
3. **E 的分解**：SVD 分解得到 4 组 (R, t) 解
   - 深度一致性检验选择正确解
4. **线性三角化**：DLT 方法，SVD 求解齐次线性方程组
5. **重投影误差**：评估重建精度

### 扩展练习
1. 使用多张图像实现完整的 SFM（添加增量式重建）
2. 实现光束法平差（Bundle Adjustment）优化
3. 添加颜色信息到点云
4. 使用自己拍摄的立体图像对进行实验
5. 对比 SIFT 和 ORB 在匹配效果上的差异

### 思考题
- 为什么需要内参数矩阵 K？如果不知道内参数会怎样？
- 三角化时为什么要检查深度（z > 0）？
- 线性三角化和非线性三角化（最小化重投影误差）有何区别？
- 如何判断重建结果的准确性？


---

## 📝 练习：手写立体视觉三维重建


**练习目标**：实现立体视觉的核心计算步骤。

**要求**：
1. 手写实现本征矩阵的计算
2. 手写实现基础矩阵的计算
3. 手写实现极线校正
4. 手写实现三维点云重建


**💡 小提示**：
- 使用 `cv_imread` / `cv_imwrite` 处理中文路径
- 除 OpenCV 读写函数外，其余代码全部手写
- 注意处理图像边界和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

def compute_F_manual(pts1, pts2):
    n = len(pts1)
    A = np.zeros((n, 9))
    for i in range(n):
        x1, y1 = pts1[i]
        x2, y2 = pts2[i]
        A[i] = [x1*x2, x1*y2, x1, y1*x2, y1*y2, y1, x2, y2, 1]
    _, _, Vt = np.linalg.svd(A)
    F = Vt[-1].reshape(3, 3)
    return F / F[2, 2]

def compute_E_manual(K, F):
    return K.T @ F @ K

def triangulate_manual(P1, P2, pts1, pts2):
    pts3d = []
    for i in range(len(pts1)):
        x1, y1 = pts1[i]
        x2, y2 = pts2[i]
        A = np.zeros((4, 4))
        A[0] = x1*P1[2,:] - P1[0,:]
        A[1] = y1*P1[2,:] - P1[1,:]
        A[2] = x2*P2[2,:] - P2[0,:]
        A[3] = y2*P2[2,:] - P2[1,:]
        _, _, Vt = np.linalg.svd(A)
        X = Vt[-1]
        pts3d.append(X[:3] / X[3])
    return np.array(pts3d)

img1 = cv_imread('stereo_image_1.jpg', cv2.IMREAD_GRAYSCALE)
img2 = cv_imread('stereo_image_2.jpg', cv2.IMREAD_GRAYSCALE)
if img1 is not None and img2 is not None:
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)
    bf = cv2.BFMatcher(cv2.NORM_L2)
    matches = bf.knnMatch(des1, des2, k=2)
    good = [m for m, n in matches if m.distance < 0.7 * n.distance]
    pts1 = np.float32([kp1[m.queryIdx].pt for m in good])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in good])
    F_m = compute_F_manual(pts1[:8], pts2[:8])
    F_cv, _ = cv2.findFundamentalMat(pts1, pts2, cv2.FM_8POINT)
    print(f'手写基础矩阵:\n{F_m}')
    print(f'OpenCV 基础矩阵:\n{F_cv}')
    K = np.array([[500,0,320],[0,500,240],[0,0,1]])
    P1 = K @ np.hstack([np.eye(3), np.zeros((3,1))])
    P2 = K @ np.hstack([np.eye(3), np.array([[-100],[0],[0]])])
    pts3d = triangulate_manual(P1, P2, pts1[:8], pts2[:8])
    print(f'3D 点:\n{pts3d}')
    print('立体视觉重建完成！')
else:
    print('立体图像对不存在')



### 💻 代码要点解释

1. **数据准备**：加载测试图像，转换数据类型

2. **算法实现**：手写核心逻辑，逐步实现每个步骤

3. **对比验证**：与 OpenCV 对应函数结果进行数值对比

4. **结果可视化**：保存处理结果，观察效果差异

5. **扩展思考**：尝试不同参数，观察算法表现

---

</details>

---
